In [3]:
from pathlib import Path
from needle_bridge import NeedleBridgeConfig, build_needle_run_plan

# -------------------------
# CONFIG
# -------------------------
cfg = NeedleBridgeConfig(
    matches_index_csv=Path("/media/hello/Vault/Tribunals/_Matches/_matches_index.csv"),
    match_frequencies_csv=None,  # not used (aggregate file)
    ws_enhanced_csv=Path("/home/hello/Projects/Statements/output/Leonardo_WS_enhanced.csv"),
    y_inferred_json=Path("/home/hello/Projects/Statements/output/Y_inferred.json"),

    # These match your real data
    et_path_col="path",
    ws_row_id_col="X1",
    out_row_id_col="row_id",
    y_row_prefix="X1_",
    y_row_pad=4,

    # Recommended filters
    filter_ws_any_needle=True,
    filter_et_any_needle=True,
)

# -------------------------
# BUILD RUN PLAN
# -------------------------
out = build_needle_run_plan(cfg)

et_df = out["et_df"]
ws_df = out["ws_df"]
row_x_tests_df = out["row_x_tests_df"]
run_plan_df = out["run_plan_df"]

# -------------------------
# GROUP (avoid re-opening same PDF per X-test)
# -------------------------
def _pack_tests(g):
    seen = set()
    out = []
    for _, r in g.iterrows():
        k = r["x_key"]
        if k in seen:
            continue
        seen.add(k)
        out.append({"x_key": k, "x_name": r["x_name"], "x_scope": r["x_scope"]})
    return out

grouped_jobs_df = (
    run_plan_df
    .groupby(["row_id", "y_row_id", "et_path", "match_mode"], dropna=False)
    .apply(lambda g: __import__("pandas").Series({  # avoids adding a new top-level import
        "matched_needles": sorted(set(g["matched_needle"])),
        "x_tests": _pack_tests(g),
    }))
    .reset_index()
)

# -------------------------
# QUICK DIAGNOSTICS
# -------------------------
print("ET docs:", len(et_df))
print("WS rows:", len(ws_df))
print("Y x_tests rows:", len(row_x_tests_df))
print("Run plan rows (raw):", len(run_plan_df))
print("Run plan jobs (grouped):", len(grouped_jobs_df))
print()

print("Unique WS rows in grouped:", grouped_jobs_df["row_id"].nunique())
print("Unique ET docs in grouped:", grouped_jobs_df["et_path"].nunique())
print()

grouped_jobs_df.head(20)

ET docs: 5062
WS rows: 3
Y x_tests rows: 55
Run plan rows (raw): 27540
Run plan jobs (grouped): 5242

Unique WS rows in grouped: 3
Unique ET docs in grouped: 2577



/tmp/ipykernel_467665/2846791038.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: __import__("pandas").Series({  # avoids adding a new top-level import


,row_id,y_row_id,et_path,match_mode,matched_needles,x_tests
0,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/1301919_...,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
1,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/1600546....,NEEDLE_OVERLAP,[has__predetermination],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
2,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/1600681....,NEEDLE_OVERLAP,[has__predetermination],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
3,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/2501187-...,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
4,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/4101776....,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
5,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/4111228....,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
6,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/8000337....,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
7,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/8000760....,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
8,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/Ali_v_RM...,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."
9,10,X1_0010,/media/hello/Vault/Tribunals/ET_Cases/Chauhan_...,NEEDLE_OVERLAP,[has__appeal_scope_regex],"[{'x_key': 'X1', 'x_name': 'Premature Disclosu..."


In [4]:
import sys, json, importlib, time
from pathlib import Path
from pprint import pprint
from tqdm.auto import tqdm

# ==========================================================
# MOLTIE RUNNER (GROUPED JOBS)
#  - consumes grouped_jobs_df (preferred) or run_plan_df (auto-groups)
#  - parses each PDF once per (row_id, et_path), runs all x_tests in-memory
# ==========================================================

# -------------------------
# REQUIRE: grouped_jobs_df OR run_plan_df exists
# -------------------------
if "grouped_jobs_df" in globals():
    jobs_df = grouped_jobs_df.copy()
elif "run_plan_df" in globals():
    # auto-group from run_plan_df
    required_cols = {"row_id", "y_row_id", "et_path", "x_key", "x_name", "x_scope", "matched_needle", "match_mode"}
    missing = required_cols - set(run_plan_df.columns)
    assert not missing, f"run_plan_df missing columns (for auto-group): {missing}"

    import pandas as pd

    def _pack_tests(g):
        seen = set()
        out = []
        for _, r in g.iterrows():
            k = r["x_key"]
            if k in seen:
                continue
            seen.add(k)
            out.append({"x_key": k, "x_name": r.get("x_name", k), "x_scope": r.get("x_scope", "GENERAL")})
        return out

    jobs_df = (
        run_plan_df
        .groupby(["row_id", "y_row_id", "et_path", "match_mode"], dropna=False)
        .apply(lambda g: pd.Series({
            "matched_needles": sorted(set(g["matched_needle"])),
            "x_tests": _pack_tests(g),
        }))
        .reset_index()
    )
else:
    raise AssertionError("Missing grouped_jobs_df or run_plan_df. Build it with the Needle Bridge first.")

required_job_cols = {"row_id", "y_row_id", "et_path", "match_mode", "matched_needles", "x_tests"}
missing = required_job_cols - set(jobs_df.columns)
assert not missing, f"jobs_df missing columns: {missing}"

# -------------------------
# USER CONTROLS
# -------------------------
DEBUG = False

# DEBUG selection: can be int, list[int], or slice
PLAN_ILOC = [3]  # examples: 3, [3], [2,5,9], slice(0,3)

# Force single PDF in DEBUG mode
FORCE_PDF = Path("/home/hello/Projects/Statements/code/appeals/1._Mr_G_Lepiarz__2._Mr_D_Lewis_v__Trades_Union_Congress_-_2200228-2023___2200230-2023.pdf")

# Batch caps (only used when DEBUG=False)
ONLY_X_KEY = None              # e.g. "X3" -> keep only that test inside x_tests list
MAX_JOBS = None                # e.g. 2000 -> cap number of PDFs/jobs (recommended)
# (Your old MAX_DOCS_PER_YROW_AND_X is less meaningful now; grouping changed the unit of work.)

# Output
REPO_ROOT = Path("/home/hello/Projects/Statements").resolve()
OUT_DIR = REPO_ROOT / "output" / "moltie_batch"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Repo / code path / Y path
# -------------------------
CODE_ROOT = (REPO_ROOT / "code").resolve()
Y_PATH = REPO_ROOT / "output" / "Y_inferred.json"

assert CODE_ROOT.exists(), f"Missing CODE_ROOT: {CODE_ROOT}"
assert Y_PATH.exists(), f"Missing Y: {Y_PATH}"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

# -------------------------
# Imports (reload once)
# -------------------------
import moltie.schemas.run_config as rc_mod
import moltie.llm.verifier_prompt as vp_mod
import moltie.llm.client as client_mod
import moltie.schemas.query_object as qo_mod
import moltie.agent.loop as loop_mod

importlib.reload(rc_mod)
importlib.reload(vp_mod)
importlib.reload(client_mod)
importlib.reload(qo_mod)
importlib.reload(loop_mod)

RunConfig = rc_mod.RunConfig
AtomQuery = qo_mod.AtomQuery
run_agent_on_one_doc = loop_mod.run_agent_on_one_doc
LLMClientConfig = client_mod.LLMClientConfig

# -------------------------
# Load Y once
# -------------------------
y_root = json.loads(Y_PATH.read_text(encoding="utf-8"))
rows = y_root.get("rows") or {}
assert isinstance(rows, dict) and rows, "Y has no rows"

# -------------------------
# Build plan_jobs (DEBUG override vs batch)
# -------------------------
if DEBUG:
    assert FORCE_PDF.exists(), f"Forced PDF missing: {FORCE_PDF}"

    if isinstance(PLAN_ILOC, slice):
        plan_jobs = jobs_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, list):
        plan_jobs = jobs_df.iloc[PLAN_ILOC].copy()
    elif isinstance(PLAN_ILOC, int):
        plan_jobs = jobs_df.iloc[[PLAN_ILOC]].copy()
    else:
        raise ValueError("PLAN_ILOC must be int, list, or slice")

    plan_jobs["et_path"] = str(FORCE_PDF)
else:
    plan_jobs = jobs_df.copy()

    if ONLY_X_KEY is not None:
        def _filter_tests(lst):
            out = [t for t in (lst or []) if t.get("x_key") == ONLY_X_KEY]
            return out
        plan_jobs["x_tests"] = plan_jobs["x_tests"].apply(_filter_tests)
        plan_jobs = plan_jobs[plan_jobs["x_tests"].apply(lambda x: len(x or []) > 0)].copy()

    if MAX_JOBS is not None:
        plan_jobs = plan_jobs.head(int(MAX_JOBS)).copy()

plan_jobs = plan_jobs.reset_index(drop=True)

print("DEBUG:", DEBUG)
print("Jobs:", len(plan_jobs))
print("Unique y_row_id:", plan_jobs["y_row_id"].nunique(), "| unique PDFs:", plan_jobs["et_path"].nunique())
print(plan_jobs[["row_id", "y_row_id", "match_mode", "et_path"]].head(10).to_string(index=False))

# -------------------------
# Client cfg (pinned once)
# -------------------------
_client_kwargs = dict(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=1000,
    max_retries=2,
)

# optional stop/debug fields
try:
    if "stop" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["stop"] = []
except Exception:
    pass

try:
    if "debug" in getattr(LLMClientConfig, "__annotations__", {}):
        _client_kwargs["debug"] = DEBUG
except Exception:
    pass

client_cfg = LLMClientConfig(**_client_kwargs)

# -------------------------
# RunConfig (pinned once)
# -------------------------
cfg2 = RunConfig.from_dict({
    "debug": DEBUG,
    "harvest_mode": False,
    "max_iters": 2,
    "window_size": 12,
    "stride": 12,
    "top_windows": 1,
    "k_chunks_per_doc": 4,
    "anchors_required": 1,
    "min_hits": 1,
    "thresh_score": 1,
    "thresh_conf": 0.8,
    "plateau_p": 1,
    "eps_improve": 0,
    "iter_temp_enabled": False,
})

# -------------------------
# PDF -> paras cache
# -------------------------
paras_cache = {}  # pdf_path_str -> paras(list[dict])

def get_paras(pdf_path: Path):
    k = str(pdf_path)
    if k in paras_cache:
        return paras_cache[k]

    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
    text = "\n".join([(p.extract_text() or "") for p in reader.pages]).strip()
    paras = [{"para_id": "p00001", "text": text}]
    paras = loop_mod._maybe_rechunk_single_blob_paras(paras)

    paras_cache[k] = paras
    return paras

# -------------------------
# Output JSONL
# -------------------------
ts = time.strftime("%Y%m%d_%H%M%S")
OUT_JSONL = OUT_DIR / f"batch_results_{'DEBUG' if DEBUG else 'BATCH'}_{ts}.jsonl"
print("\nWriting results to:", OUT_JSONL)

def safe_to_dict(obj):
    if obj is None:
        return None
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    if isinstance(obj, dict):
        return obj
    return {"repr": repr(obj)}

n_ok = 0
n_neg = 0
n_err = 0

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for j, job in tqdm(plan_jobs.iterrows(), total=len(plan_jobs), desc="moltie jobs", unit="job"):
        y_row_id = job["y_row_id"]
        pdf_path = Path(job["et_path"])
        row_id = job.get("row_id")
        match_mode = job.get("match_mode")
        matched_needles = job.get("matched_needles") or []

        try:
            if y_row_id not in rows:
                raise KeyError(f"y_row_id not in Y.rows: {y_row_id}")
            if not pdf_path.exists():
                raise FileNotFoundError(f"PDF missing: {pdf_path}")

            # row-scoped y object
            y_obj = (rows[y_row_id] or {}).get("y") or {}

            # parse / rechunk ONCE per PDF
            paras = get_paras(pdf_path)
            doc_id = pdf_path.stem

            # run all x_tests for this job
            for t in (job.get("x_tests") or []):
                x_key = t.get("x_key")
                x_name = t.get("x_name", x_key)

                try:
                    merged = qo_mod.merge_indicators_and_excludes(y_obj, [x_key])
                    atom = AtomQuery(
                        atom_id=x_key,
                        x_tests=[x_key],
                        proposition=x_name,
                        positive_indicators=merged["positive_indicators"],
                        excludes=merged["excludes"],
                        keyword_seeds=merged["positive_indicators"],
                        expansion_terms=[],
                    )

                    res = run_agent_on_one_doc(doc_id, paras, atom, cfg2, client_cfg)

                    verdict = safe_to_dict(getattr(res, "verdict", None))
                    negative_exit = safe_to_dict(getattr(res, "negative_exit", None))

                    if verdict:
                        n_ok += 1
                    else:
                        n_neg += 1

                    row_out = {
                        "job_i": int(j),
                        "row_id": row_id,
                        "y_row_id": y_row_id,
                        "match_mode": match_mode,
                        "matched_needles": matched_needles,
                        "et_path": str(pdf_path),
                        "doc_id": doc_id,
                        "x_key": x_key,
                        "x_name": x_name,
                        "verdict": verdict,
                        "negative_exit": negative_exit,
                        "iters": getattr(res, "iters", None),
                        "trace_tail": (getattr(res, "trace", None) or [])[-3:],
                    }
                    f.write(json.dumps(row_out, ensure_ascii=False) + "\n")

                    if DEBUG:
                        print("\n--- JOB", j, "| X", x_key, "---")
                        print("y_row_id:", y_row_id, "| pdf:", pdf_path.name)
                        if verdict:
                            print("relevant=", verdict.get("relevant"),
                                  "score=", verdict.get("precedent_score"),
                                  "conf=", verdict.get("confidence"),
                                  "anchors=", len(verdict.get("anchors") or []))
                        else:
                            print("NEGATIVE:", (negative_exit or {}).get("reason"))

                except Exception as e_x:
                    n_err += 1
                    f.write(json.dumps({
                        "job_i": int(j),
                        "row_id": row_id,
                        "y_row_id": y_row_id,
                        "match_mode": match_mode,
                        "et_path": str(pdf_path),
                        "doc_id": doc_id,
                        "x_key": x_key,
                        "error": repr(e_x),
                    }, ensure_ascii=False) + "\n")

        except Exception as e_job:
            n_err += 1
            f.write(json.dumps({
                "job_i": int(j),
                "row_id": row_id,
                "y_row_id": y_row_id,
                "match_mode": match_mode,
                "et_path": str(pdf_path),
                "error": repr(e_job),
            }, ensure_ascii=False) + "\n")

print("\nDONE")
print("ok:", n_ok, "| negative:", n_neg, "| errors:", n_err)
print("results:", OUT_JSONL)

DEBUG: False
Jobs: 5242
Unique y_row_id: 3 | unique PDFs: 2577
 row_id y_row_id     match_mode                                                                                                                                     et_path
     10  X1_0010 NEEDLE_OVERLAP                                         /media/hello/Vault/Tribunals/ET_Cases/1301919_17_Mr_P_Eboye_v_Farsight_UK_Security_Services_Ltd.pdf
     10  X1_0010 NEEDLE_OVERLAP              /media/hello/Vault/Tribunals/ET_Cases/1600546.2016_Miss_Helen_Evans_v_Llanishen_Fach_Primary_School_-_Judgment_and_Reasons.pdf
     10  X1_0010 NEEDLE_OVERLAP /media/hello/Vault/Tribunals/ET_Cases/1600681.2016_Mr_Stephen_Curtis_-v-_British_Association_of_Shooting_Reserved_Judgment_with_reasons.pdf
     10  X1_0010 NEEDLE_OVERLAP                                                              /media/hello/Vault/Tribunals/ET_Cases/2501187-16_Howie_-_Reserved_Judgment.pdf
     10  X1_0010 NEEDLE_OVERLAP                             /media/hello/Vaul

moltie jobs:   0%|          | 0/5242 [00:00<?, ?job/s]


DONE
ok: 7582 | negative: 19270 | errors: 0
results: /home/hello/Projects/Statements/output/moltie_batch/batch_results_BATCH_20260226_053057.jsonl


In [ ]:
from pathlib import Path
import json
import pandas as pd

# =========================================================
# MASTER MOLTIE -> CSV PIPELINE
# 1) Load MASTER_moltie.jsonl
# 2) Promote negative_exit.best_attempt when verdict is null
# 3) Flatten key verdict fields
# 4) Split into relevant / remaining
# 5) Sort relevant by (anchor_count desc, confidence desc)
# 6) Save:
#    - MASTER_moltie.csv (all rows flattened)
#    - MASTER_moltie__relevant.csv
#    - MASTER_moltie__remaining.csv
# 7) Collapse by et_path (one row per PDF), compress x_key/x_name with "||"
#    - MASTER_moltie__relevant__by_pdf.csv
#    - MASTER_moltie__remaining__by_pdf.csv
# =========================================================

# -------------------------
# INPUT
# -------------------------
JSONL_PATH = Path("/home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie.jsonl")
print("Loading:", JSONL_PATH)
assert JSONL_PATH.exists(), f"Missing: {JSONL_PATH}"

# -------------------------
# LOAD JSONL
# -------------------------
rows = []
with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
print("Raw shape:", df.shape)

# -------------------------
# PROMOTE best_attempt IF verdict is None
# -------------------------
def effective_verdict(row):
    v = row.get("verdict")
    if isinstance(v, dict):
        return v
    neg = row.get("negative_exit")
    if isinstance(neg, dict):
        best = neg.get("best_attempt")
        if isinstance(best, dict):
            return best
    return None

df["effective_verdict"] = df.apply(effective_verdict, axis=1)

# -------------------------
# FLATTEN verdict fields
# -------------------------
def extract_field(d, key):
    if isinstance(d, dict):
        return d.get(key)
    return None

df["relevant"] = df["effective_verdict"].apply(lambda v: extract_field(v, "relevant"))
df["precedent_score"] = df["effective_verdict"].apply(lambda v: extract_field(v, "precedent_score"))
df["confidence"] = df["effective_verdict"].apply(lambda v: extract_field(v, "confidence"))
df["matched_X"] = df["effective_verdict"].apply(lambda v: extract_field(v, "matched_X"))

df["anchor_count"] = df["effective_verdict"].apply(
    lambda v: len(v.get("anchors") or []) if isinstance(v, dict) else 0
)

# Optional extra diagnostics (useful later)
df["negative_reason"] = df.get("negative_exit").apply(
    lambda v: v.get("reason") if isinstance(v, dict) else None
) if "negative_exit" in df.columns else None

# -------------------------
# SELECT CLEAN COLUMNS
# -------------------------
final_cols = [
    "row_id",
    "y_row_id",
    "x_key",
    "x_name",
    "et_path",
    "relevant",
    "precedent_score",
    "confidence",
    "anchor_count",
    "negative_reason",
]

# Keep only columns that exist (defensive)
final_cols = [c for c in final_cols if c in df.columns]

df_out = df[final_cols].copy()
print("Flattened shape:", df_out.shape)

# -------------------------
# SPLIT: relevant vs remaining
# -------------------------
df_relevant = df_out[df_out["relevant"] == True].copy()
df_remaining = df_out[df_out["relevant"] != True].copy()

# Sort relevant: strongest anchors first, then confidence
df_relevant = df_relevant.sort_values(
    by=["anchor_count", "confidence"],
    ascending=[False, False],
    na_position="last"
)

print("Relevant shape:", df_relevant.shape)
print("Remaining shape:", df_remaining.shape)

# -------------------------
# SAVE CSVs (row-level)
# -------------------------
BASE_NAME = JSONL_PATH.with_suffix("")  # Path without ".jsonl"

CSV_ALL = BASE_NAME.with_suffix(".csv")
CSV_RELEVANT = BASE_NAME.with_name(BASE_NAME.name + "__relevant.csv")
CSV_REMAINING = BASE_NAME.with_name(BASE_NAME.name + "__remaining.csv")

df_out.to_csv(CSV_ALL, index=False)
df_relevant.to_csv(CSV_RELEVANT, index=False)
df_remaining.to_csv(CSV_REMAINING, index=False)

print("Saved ALL:", CSV_ALL)
print("Saved RELEVANT:", CSV_RELEVANT)
print("Saved REMAINING:", CSV_REMAINING)

# -------------------------
# DEDUPE BY et_path (one row per PDF)
# compress Xs using "||"
# -------------------------
SEP = "||"

def _compress_unique_in_order(vals):
    out = []
    seen = set()
    for v in vals:
        if v is None:
            continue
        v = str(v)
        if v in seen:
            continue
        seen.add(v)
        out.append(v)
    return SEP.join(out)

def collapse_by_pdf(df_in: pd.DataFrame) -> pd.DataFrame:
    if df_in.empty:
        return pd.DataFrame(columns=[
            "et_path",
            "n_atoms",
            "max_anchor_count",
            "max_confidence",
            "max_precedent_score",
            "x_keys_comp",
            "x_names_comp",
            "best_row_id",
            "best_y_row_id",
        ])

    # Ensure sorted so "best" Xs appear first in the compression
    df_sorted = df_in.sort_values(
        by=["et_path", "anchor_count", "confidence"],
        ascending=[True, False, False],
        na_position="last"
    )

    rows_out = []
    for et_path, g in df_sorted.groupby("et_path", dropna=False):
        # g is already sorted best-first for this et_path
        rows_out.append({
            "et_path": et_path,
            "n_atoms": int(len(g)),
            "max_anchor_count": int(g["anchor_count"].fillna(0).max()),
            "max_confidence": float(g["confidence"].fillna(0).max()),
            "max_precedent_score": float(g["precedent_score"].fillna(0).max()) if "precedent_score" in g.columns else 0.0,
            "x_keys_comp": _compress_unique_in_order(g["x_key"].tolist()) if "x_key" in g.columns else "",
            "x_names_comp": _compress_unique_in_order(g["x_name"].tolist()) if "x_name" in g.columns else "",
            "best_row_id": g.iloc[0]["row_id"] if "row_id" in g.columns else None,
            "best_y_row_id": g.iloc[0]["y_row_id"] if "y_row_id" in g.columns else None,
        })

    out = pd.DataFrame(rows_out)

    # Sort PDFs by strongest evidence
    out = out.sort_values(
        by=["max_anchor_count", "max_confidence", "n_atoms"],
        ascending=[False, False, False],
        na_position="last"
    ).reset_index(drop=True)

    return out

df_relevant_pdf = collapse_by_pdf(df_relevant)
df_remaining_pdf = collapse_by_pdf(df_remaining)

# -------------------------
# SAVE CSVs (pdf-level)
# -------------------------
CSV_RELEVANT_PDF = BASE_NAME.with_name(BASE_NAME.name + "__relevant__by_pdf.csv")
CSV_REMAINING_PDF = BASE_NAME.with_name(BASE_NAME.name + "__remaining__by_pdf.csv")

df_relevant_pdf.to_csv(CSV_RELEVANT_PDF, index=False)
df_remaining_pdf.to_csv(CSV_REMAINING_PDF, index=False)

print("Saved RELEVANT by PDF:", CSV_RELEVANT_PDF, "| shape:", df_relevant_pdf.shape)
print("Saved REMAINING by PDF:", CSV_REMAINING_PDF, "| shape:", df_remaining_pdf.shape)

# -------------------------
# Preview
# -------------------------
print("\nTop relevant rows:")
display(df_relevant.head(10))

print("\nTop relevant PDFs:")
display(df_relevant_pdf.head(10))